# Task 3, Steps 5-6a: Adapter Merging and Distribution Alignment on Common Prefixes

* Merge each of the four LoRA adapters (off/on-policy x seeds 6304/6305) into a *fresh* FP16 0.5B backbone, save one
  merged checkpoint per run, and compare merged vs unmerged logits on a fixed prompt batch.
* Generate each trained student's **own-rollout bank** (same prompts / positions protocol, producing checkpoint named).
* On identical prefix records, compute forward KL(p||q), reverse KL(q||p), TV, greedy next-token agreement and
  expected one-token acceptance 1 - TV for the original draft and all four trained drafts, on: the teacher bank, the
  original-draft bank, the exact-speculative bank, and all four student banks (every draft is also evaluated on the
  other students' frozen rollouts). Breakdowns by workload (math / familiar / shifted extraction) and response
  position ([1,16], [17,32], [33,64], long 65-192). Token-weighted means over the same record set for every draft.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task3_speculative")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2" or _ver("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5", "peft>=0.13"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task3", need_public=True, need_prefix_banks=False)

In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.io import read_jsonl, write_jsonl, save_json, load_json
from shared.src.plotting import save_figure
from task3_speculative.src import prefix_banks as PB
from task3_speculative.src.data import test_prompts
from task3_speculative.src.distill import load_models, merge_adapter, load_merged

M = load_models()
tok, target, original = M["tok"], M["target"], M["draft"]
RUNS = ctx.checkpoints / "distill"
MERGED = ctx.checkpoints / "merged"
BANKS = ctx.checkpoints / "banks"
OUT = ctx.results / "alignment"
OUT.mkdir(exist_ok=True)
STUDENTS = [f"{m}_s{s}" for m in ("off", "on") for s in (6304, 6305)]

In [ ]:
merge_info = {}
check = test_prompts()[:4] + test_prompts()[60:64]
for name in STUDENTS:
    dest = MERGED / name
    info_f = ctx.results / "distill" / f"merge_{name}.json"
    if not (dest / "config.json").exists():
        method, seed = name.split("_s")
        merge_info[name] = merge_adapter(RUNS / f"{method}_policy_s{seed}" / "adapter", dest, tok, check)
        save_json(info_f, merge_info[name])
    merge_info[name] = load_json(info_f)
pd.DataFrame({k: {kk: v[kk] for kk in ("merge_time_s", "max_abs_logit_diff", "min_argmax_agree")} for k, v in merge_info.items()}).T

In [ ]:
drafts = {"original": original}
for name in STUDENTS:
    drafts[name] = load_merged(MERGED / name)
for name in STUDENTS:
    f = BANKS / f"student_{name}.jsonl.gz"
    if not f.exists():
        write_jsonl(f, PB.build_bank(f"student_{name}", tok, model=drafts[name], producer=f"merged {name}"))
bank_files = ["teacher", "original_draft", "exact_spec_original_gamma4"] + [f"student_{n}" for n in STUDENTS]
records = {b: read_jsonl(BANKS / f"{b}.jsonl.gz") for b in bank_files}
print({b: len(r) for b, r in records.items()})

In [ ]:
met_path = OUT / "prefix_metrics.jsonl.gz"
if not met_path.exists():
    rows = []
    for b in bank_files:
        rows += PB.bank_metrics(target, drafts, tok, records[b])
        print("done", b, flush=True)
        gc.collect(); torch.cuda.empty_cache()
    write_jsonl(met_path, rows)
met = pd.DataFrame(read_jsonl(met_path))
met["method"] = met.draft.str.split("_").str[0]
METRICS = ["fkl", "rkl", "tv", "greedy_agree", "exp_accept"]
main = met.groupby(["bank", "draft"])[METRICS].mean().unstack("draft").round(4)
main.to_csv(OUT / "alignment_by_bank.csv")
main

## Off- vs on-policy (seed mean and spread) on shared banks

In [ ]:
seedwise = met.groupby(["bank", "method", "draft"])[METRICS].mean().reset_index()
disp = seedwise.groupby(["bank", "method"])[METRICS].agg(["mean", "std"]).round(4)
disp.to_csv(OUT / "alignment_seed_dispersion.csv")
disp.xs("exp_accept", axis=1, level=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sub = met[met.bank.isin(["teacher", "exact_spec_original_gamma4", "original_draft"])]
sns.barplot(data=sub, x="workload", y="exp_accept", hue="draft", ax=axes[0], errorbar=None)
axes[0].set_title("Expected one-token acceptance 1 - TV by workload (shared banks)")
sns.barplot(data=sub, x="pos_bin", y="exp_accept", hue="draft", ax=axes[1], errorbar=None,
            order=["1-16", "17-32", "33-64", "65-192"])
axes[1].set_title("... by response position")
save_figure(ctx.figures, "alignment_by_workload_and_position")
met.pivot_table(index=["workload", "pos_bin"], columns="draft", values="exp_accept").round(3)

## Student-rollout banks: each draft on every student's own rollouts (exposure diagnostic)

In [ ]:
own = met[met.bank.str.startswith("student_")].pivot_table(index="bank", columns="draft", values=["exp_accept", "fkl"]).round(4)
own.to_csv(OUT / "alignment_student_banks.csv")
own

## Aligned prefix-level disagreements (fixed rule)
Rule: on the teacher bank, the 8 records with the largest increase in TV of the on-policy s6304 draft over the
off-policy s6304 draft, and the 8 with the largest decrease.

In [ ]:
t = met[met.bank == "teacher"].pivot_table(index=["prompt_id", "position"], columns="draft", values="tv")
t["d"] = t["on_s6304"] - t["off_s6304"]
pick = pd.concat([t.nlargest(8, "d"), t.nsmallest(8, "d")])
tb = {(r["prompt_id"], r["position"]): r for r in records["teacher"]}
rows = []
for (pid, pos), r in pick.iterrows():
    rec = tb[(pid, pos)]
    ctx_txt = tok.decode(rec["response_prefix_token_ids"][-25:])
    rows.append(dict(prompt_id=pid, position=pos, prefix_tail=ctx_txt.replace("\n", " "), tv_off=r["off_s6304"], tv_on=r["on_s6304"], tv_original=r["original"]))
dis = pd.DataFrame(rows)
dis.to_csv(OUT / "aligned_disagreements.csv", index=False)
dis